In [4]:
"""
Sepsis Prediction — LSTM v2 (Improved)
=======================================
Fixes applied vs v1:
  1. WeightedRandomSampler  → fixes class imbalance at the data level
  2. pos_weight in loss     → doubly penalises missed sepsis predictions
  3. Bidirectional LSTM     → captures patterns in both temporal directions
  4. Attention mechanism    → model focuses on relevant timesteps, not just last one
  5. Gradient clipping      → prevents exploding gradients over 336 timesteps
  6. LR scheduler           → reduces LR when AUROC plateaus
  7. Early stopping         → stops before overfitting, restores best weights
  8. Stratified split check → prints class rates so you can verify split is correct
  9. Full experiment logger → saves every run to results/model_log.csv
 10. Threshold tuning via   → precision-recall curve (more reliable than F1 sweep)
     precision-recall curve
"""

import os
import csv
import numpy as np
import pandas as pd
from datetime import datetime
from sklearn.metrics import (
    classification_report,
    f1_score,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.model_selection import ParameterGrid
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these paths to match your folder layout
# ─────────────────────────────────────────────────────────────────────────────
PARENT_DIR       = ".."
TRAIN_FOLDER     = os.path.join(PARENT_DIR, "processed_training")
VAL_FOLDER       = os.path.join(PARENT_DIR, "processed_training_setB")
HEADER_FOLDER    = os.path.join(PARENT_DIR, "processed_training_Ay")
SAVE_DIR         = "saved_models"
RESULTS_DIR      = "results"
MODEL_VERSION    = "lstm_v2_bidirectional_attention"

EARLY_HOURS      = 6        # predict sepsis this many hours before onset
MAX_TIMESTEPS    = 336      # truncate / pad sequences to this length
EPOCHS           = 50       # max epochs (early stopping will cut this short)
BATCH_SIZE       = 64       # larger batch = more stable gradients
PATIENCE         = 10       # early stopping patience (epochs without improvement)

os.makedirs(SAVE_DIR,   exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Device setup
# ─────────────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Feature columns from header folder
# ─────────────────────────────────────────────────────────────────────────────
all_columns = set()
if not os.path.exists(HEADER_FOLDER):
    raise FileNotFoundError(f"Header folder not found: {os.path.abspath(HEADER_FOLDER)}")

for f in os.listdir(HEADER_FOLDER):
    if f.endswith(".psv"):
        with open(os.path.join(HEADER_FOLDER, f), "r") as fh:
            header = fh.readline().strip()
            if header:
                all_columns.update(header.split("|"))

FEATURE_COLUMNS = sorted(c for c in all_columns if c != "SepsisLabel")
print(f"Feature count: {len(FEATURE_COLUMNS)}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Data loading & padding
# ─────────────────────────────────────────────────────────────────────────────
def load_all_patients(folders, feature_columns, early_hours=6):
    """Load from multiple folders into one unified dataset."""
    X, y = [], []
    for folder in folders:
        if not os.path.exists(folder):
            print(f"  Skipping missing folder: {folder}")
            continue
        for f in sorted(os.listdir(folder)):
            if not f.endswith(".psv"):
                continue
            df = pd.read_csv(os.path.join(folder, f), sep="|")
            df = df.reindex(columns=feature_columns + ["SepsisLabel"])
            df[feature_columns] = df[feature_columns].fillna(0.0)

            features = df[feature_columns].values.astype(np.float32)
            labels   = df["SepsisLabel"].values

            if labels.max() == 1:
                t_sepsis = np.where(labels == 1)[0][0]
                cutoff   = t_sepsis - early_hours
                if cutoff <= 0:
                    continue
                X.append(features[:cutoff])
                y.append(1)
            else:
                X.append(features)
                y.append(0)

    return X, np.array(y, dtype=np.int64)


def pad_sequences(X, max_len=None):
    """Pre-pad (zeros at start) and pre-truncate sequences to max_len."""
    if max_len is None:
        max_len = max(len(x) for x in X)
    n_features = X[0].shape[1]
    X_pad = np.zeros((len(X), max_len, n_features), dtype=np.float32)
    for i, seq in enumerate(X):
        L = len(seq)
        if L >= max_len:
            X_pad[i] = seq[-max_len:]           # keep most recent timesteps
        else:
            X_pad[i, max_len - L:] = seq        # pad at beginning
    return X_pad, max_len


# ─────────────────────────────────────────────────────────────────────────────
# 4. Load data
# ─────────────────────────────────────────────────────────────────────────────
# Combine ALL folders into one pool
ALL_FOLDERS = [
    os.path.join(PARENT_DIR, "processed_training"),
    os.path.join(PARENT_DIR, "processed_training_setB"),
    os.path.join(PARENT_DIR, "processed_training_Ay"),  # include if it has labels
]

print("Loading all patient data...")
X_all, y_all = load_all_patients(ALL_FOLDERS, FEATURE_COLUMNS)
print(f"Total patients: {len(y_all)} | Sepsis: {y_all.sum()} ({y_all.mean():.3f})")

# Stratified split — guarantees same sepsis rate in train and val
indices = np.arange(len(y_all))
idx_train, idx_val = train_test_split(
    indices,
    test_size    = 0.2,
    stratify     = y_all,   # <-- this is the key line
    random_state = 42
)

X_train_raw = [X_all[i] for i in idx_train]
X_val_raw   = [X_all[i] for i in idx_val]
y_train      = y_all[idx_train]
y_val        = y_all[idx_val]

print(f"Train: {len(y_train)} | sepsis rate: {y_train.mean():.3f}")
print(f"Val:   {len(y_val)}   | sepsis rate: {y_val.mean():.3f}")
# ─────────────────────────────────────────────────────────────────────────────
# 5. Class imbalance — compute weights
# ─────────────────────────────────────────────────────────────────────────────
class_counts  = np.bincount(y_train)              # [n_negative, n_positive]
imbalance_ratio = class_counts[0] / class_counts[1]
print(f"\nClass counts — 0: {class_counts[0]}, 1: {class_counts[1]}")
print(f"Imbalance ratio: {imbalance_ratio:.1f}:1")

# For WeightedRandomSampler: give each sample a weight inverse to its class frequency
sample_weights = np.where(y_train == 1,
                          1.0 / class_counts[1],
                          1.0 / class_counts[0])

# For BCEWithLogitsLoss pos_weight: how much more to penalise false negatives
pos_weight = torch.tensor([imbalance_ratio], dtype=torch.float32).to(device)

# ─────────────────────────────────────────────────────────────────────────────
# 6. DataLoaders
# ─────────────────────────────────────────────────────────────────────────────
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train, dtype=torch.float32)
)
val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val, dtype=torch.float32)
)

# WeightedRandomSampler oversamples minority class so each batch is ~balanced
sampler = WeightedRandomSampler(
    weights     = torch.tensor(sample_weights, dtype=torch.float32),
    num_samples = len(sample_weights),
    replacement = True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# ─────────────────────────────────────────────────────────────────────────────
# 7. Model architecture — Bidirectional LSTM + Attention
# ─────────────────────────────────────────────────────────────────────────────
class SepsisLSTM(nn.Module):
    """
    Bidirectional LSTM with a temporal attention mechanism.

    Why bidirectional: reads the sequence forwards AND backwards.
    Why attention: instead of using only the last timestep, the model
    learns which hours in the ICU stay are most predictive and weighs
    them accordingly — much more informative for 336-step sequences.
    """
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size    = input_dim,
            hidden_size   = hidden_dim,
            num_layers    = num_layers,
            batch_first   = True,
            dropout       = dropout if num_layers > 1 else 0.0,
            bidirectional = True,           # output dim = hidden_dim * 2
        )
        lstm_out_dim = hidden_dim * 2       # *2 because bidirectional

        # Attention: score each timestep, softmax to get weights, weighted sum
        self.attention = nn.Linear(lstm_out_dim, 1)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(lstm_out_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            # No sigmoid here — BCEWithLogitsLoss applies it internally (numerically stable)
        )

    def forward(self, x):
        # lstm_out: (batch, timesteps, hidden_dim*2)
        lstm_out, _ = self.lstm(x)

        # Attention scores over timesteps
        attn_scores  = self.attention(lstm_out)          # (batch, timesteps, 1)
        attn_weights = torch.softmax(attn_scores, dim=1) # normalise over time
        context      = (lstm_out * attn_weights).sum(dim=1)  # (batch, hidden_dim*2)

        logits = self.classifier(context)                # (batch, 1)
        return logits


# ─────────────────────────────────────────────────────────────────────────────
# 8. Experiment logger
# ─────────────────────────────────────────────────────────────────────────────
LOG_PATH = os.path.join(RESULTS_DIR, "model_log.csv")
LOG_FIELDS = ["timestamp", "version", "params", "epochs_run",
              "auroc", "auprc", "f1_sepsis", "recall_sepsis",
              "precision_sepsis", "threshold", "notes"]

def log_experiment(version, params, epochs_run, auroc, auprc,
                   f1, recall, precision, threshold, notes=""):
    row = {
        "timestamp":        datetime.now().strftime("%Y-%m-%d %H:%M"),
        "version":          version,
        "params":           str(params),
        "epochs_run":       epochs_run,
        "auroc":            round(auroc, 4),
        "auprc":            round(auprc, 4),
        "f1_sepsis":        round(f1, 4),
        "recall_sepsis":    round(recall, 4),
        "precision_sepsis": round(precision, 4),
        "threshold":        round(threshold, 3),
        "notes":            notes,
    }
    write_header = not os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow(row)
    print(f"  → Logged to {LOG_PATH}")


# ─────────────────────────────────────────────────────────────────────────────
# 9. Evaluation helpers
# ─────────────────────────────────────────────────────────────────────────────
def get_val_probs(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for batch_x, _ in loader:
            batch_x = batch_x.to(device)
            logits  = model(batch_x).squeeze(1)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
    return np.array(probs)


def best_threshold_by_f1(y_true, y_prob):
    """
    Use the precision-recall curve to find the threshold that maximises F1
    for the positive (sepsis) class. More reliable than a manual sweep.
    """
    precision_arr, recall_arr, thresholds = precision_recall_curve(y_true, y_prob)
    # avoid divide-by-zero
    denom = precision_arr[:-1] + recall_arr[:-1]
    denom = np.where(denom == 0, 1e-8, denom)
    f1_arr = 2 * (precision_arr[:-1] * recall_arr[:-1]) / denom
    best_idx = np.argmax(f1_arr)
    return thresholds[best_idx], f1_arr[best_idx]


# ─────────────────────────────────────────────────────────────────────────────
# 10. Training loop (with early stopping & LR scheduler)
# ─────────────────────────────────────────────────────────────────────────────
def train_model(params, model_version_tag):
    print(f"\n{'='*55}")
    print(f"Training: {params}")
    print(f"{'='*55}")

    model = SepsisLSTM(
        input_dim  = N_FEATURES,
        hidden_dim = params["hidden_dim"],
        num_layers = params["num_layers"],
        dropout    = params["dropout"],
    ).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=params["lr"], weight_decay=1e-4)

    # Halve LR if AUROC doesn't improve for 5 epochs
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=5, factor=0.5
    )

    best_auroc        = 0.0
    best_state        = None
    patience_counter  = 0
    history           = []

    for epoch in range(1, EPOCHS + 1):
        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        total_loss = 0.0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            logits = model(batch_x).squeeze(1)
            loss   = criterion(logits, batch_y)
            loss.backward()
            # Clip gradients — essential for long sequences (336 timesteps)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        # ── Validate ───────────────────────────────────────────────────────
        y_prob_val = get_val_probs(model, val_loader)
        auroc      = roc_auc_score(y_val, y_prob_val)
        avg_loss   = total_loss / len(train_loader)

        scheduler.step(auroc)
        history.append({"epoch": epoch, "loss": avg_loss, "auroc": auroc})

        marker = ""
        if auroc > best_auroc:
            best_auroc  = auroc
            best_state  = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            marker = " ✓ best"
            # Save checkpoint immediately
            ckpt_path = os.path.join(SAVE_DIR, f"{model_version_tag}_best.pt")
            torch.save(best_state, ckpt_path)
        else:
            patience_counter += 1

        print(f"  Epoch {epoch:02d}/{EPOCHS} | loss: {avg_loss:.4f} | "
              f"val AUROC: {auroc:.4f}{marker}")

        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

    # Restore best weights
    model.load_state_dict(best_state)
    return model, best_auroc, len(history)


# ─────────────────────────────────────────────────────────────────────────────
# 11. Grid search
# ─────────────────────────────────────────────────────────────────────────────
param_grid = ParameterGrid({
    "hidden_dim": [64, 128],
    "num_layers": [2],
    "dropout":    [0.3, 0.4],
    "lr":         [1e-3],
})

best_overall_auroc  = 0.0
best_overall_model  = None
best_overall_params = None
best_overall_epochs = 0

for i, params in enumerate(param_grid):
    tag   = f"{MODEL_VERSION}_run{i+1}"
    model, auroc, epochs_run = train_model(params, tag)

    if auroc > best_overall_auroc:
        best_overall_auroc  = auroc
        best_overall_model  = model
        best_overall_params = params
        best_overall_epochs = epochs_run

print(f"\n{'='*55}")
print(f"Best grid search AUROC: {best_overall_auroc:.4f}")
print(f"Best params: {best_overall_params}")

# ─────────────────────────────────────────────────────────────────────────────
# 12. Final evaluation
# ─────────────────────────────────────────────────────────────────────────────
y_prob = get_val_probs(best_overall_model, val_loader)

auroc = roc_auc_score(y_val, y_prob)
auprc = average_precision_score(y_val, y_prob)

# Find best threshold from precision-recall curve
best_t, best_f1 = best_threshold_by_f1(y_val, y_prob)
y_pred = (y_prob > best_t).astype(int)

report = classification_report(y_val, y_pred, output_dict=True)
sepsis_metrics = report.get("1", report.get("1.0", {}))

print(f"\n{'='*55}")
print("FINAL PERFORMANCE REPORT")
print(f"{'='*55}")
print(f"ROC-AUC  : {auroc:.4f}   (target ≥ 0.80)")
print(f"AUPRC    : {auprc:.4f}   (target ≥ 0.30 for imbalanced data)")
print(f"Threshold: {best_t:.3f}")
print()
print(classification_report(y_val, y_pred))

# ─────────────────────────────────────────────────────────────────────────────
# 13. Save best model + log experiment
# ─────────────────────────────────────────────────────────────────────────────
final_path = os.path.join(SAVE_DIR, f"{MODEL_VERSION}_final.pt")
torch.save(best_overall_model.state_dict(), final_path)
print(f"Model saved → {final_path}")

log_experiment(
    version    = MODEL_VERSION,
    params     = best_overall_params,
    epochs_run = best_overall_epochs,
    auroc      = auroc,
    auprc      = auprc,
    f1         = sepsis_metrics.get("f1-score", 0),
    recall     = sepsis_metrics.get("recall",   0),
    precision  = sepsis_metrics.get("precision",0),
    threshold  = best_t,
    notes      = f"Bidirectional LSTM + attention, imbalance ratio {imbalance_ratio:.1f}:1",
)

# ─────────────────────────────────────────────────────────────────────────────
# 14. Model inspection
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("Model architecture:")
print(f"{'='*55}")
print(best_overall_model)

print(f"\n{'='*55}")
print("Layer parameter shapes:")
print(f"{'='*55}")
for name, param in best_overall_model.named_parameters():
    if param.requires_grad:
        print(f"  {name:35} {tuple(param.shape)}")

# ─────────────────────────────────────────────────────────────────────────────
# 15. Quick performance summary vs v1
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print("v1 → v2 comparison")
print(f"{'='*55}")
print(f"  AUROC   : 0.617  →  {auroc:.3f}")
print(f"  F1(sep) : 0.14   →  {sepsis_metrics.get('f1-score', 0):.3f}")
print(f"  Recall  : 0.27   →  {sepsis_metrics.get('recall', 0):.3f}")
print(f"  Threshold: 0.9   →  {best_t:.3f}")

Using device: cuda
Feature count: 32
Loading all patient data...
Total patients: 59458 | Sepsis: 3508 (0.059)
Train: 47566 | sepsis rate: 0.059
Val:   11892   | sepsis rate: 0.059

Class counts — 0: 44760, 1: 2806
Imbalance ratio: 16.0:1


AssertionError: Size mismatch between tensors